# Image Clustering Using VGG16 and K-Means by Mateo Vergara

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Importing libraries, loading a pre-trained VGG16 CNN, and defining functions for preprocessing and feature extraction

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input #We're using VGG16 pre-trained CNN for feature extraction
from tensorflow.keras.preprocessing import image
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

model = VGG16(weights='imagenet', include_top=False, pooling='avg')


def load_and_preprocess_image(img_path): #Preprocessing step: resizing the images to 224x224 and converting them into arrays
    img = image.load_img(img_path, target_size=(224, 224))
    img_data = image.img_to_array(img)
    img_data = np.expand_dims(img_data, axis=0)
    img_data = preprocess_input(img_data)
    return img_data

def extract_features(img_path, model): #Extracting feature vectors from each image using the VGG16 model
    img_data = load_and_preprocess_image(img_path)
    features = model.predict(img_data)
    return features.flatten()

input_folder = "/content/drive/MyDrive/path/to/your/directory" #Edit this path to yours
image_paths = [os.path.join(input_folder, fname) for fname in os.listdir(input_folder) if fname.endswith(('.jpg', '.png', '.jpeg'))]
features = np.array([extract_features(img_path, model) for img_path in image_paths])

## Setting up the number of clusters

In [5]:
n_clusters = 3 #Number of clusters goes here, change it to your needs

kmeans = KMeans(n_clusters = n_clusters, random_state = 42) #KMeans labels the images based on the features gathered by the CNN

kmeans.fit(features)

labels = kmeans.labels_

## Clustering images into folders based on their labels

In [6]:
import shutil

output_folder = "/content/output_folder" #Change this line to your desired output path. You can leave it as it is.

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for img_path, label in zip(image_paths, labels): #This is where images are organized into folders according to their cluster label
    cluster_folder = os.path.join(output_folder, f"cluster_{label}")
    if not os.path.exists(cluster_folder):
        os.makedirs(cluster_folder)
    img_name = os.path.basename(img_path)
    shutil.copy(img_path, os.path.join(cluster_folder, img_name))

## Compressing the clustered data and downloading it

In [7]:
import shutil
from google.colab import files

shutil.make_archive("/content/output_folder", 'zip', output_folder) #Keep this line as it is if you havent't change the output_folder in block 6

files.download("/content/output_folder.zip") #The same path as the output folder but adding the .zip extension

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>